# **Imports**

In [2]:
from earthscape.constants import *

import os
import glob
import json
import pandas as pd
import geopandas as gpd
import rasterio
import numpy as np

# **Compiled Data**

# *Areas, Patch Locations, & Class Mappings*

In [26]:
######################################################################
# Compile local dataset area, label, patch files into single global files
######################################################################

# paths to local files
local_dataset_dirs = [os.path.join(DATASET_DIR, d) for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d)) and d != 'smoke']


# get all local area/patch/mapping paths...
area_paths = [glob.glob(f"{ldd}/*areas.csv")[0] for ldd in local_dataset_dirs]
patch_paths = [glob.glob(f"{ldd}/*patches.geojson")[0] for ldd in local_dataset_dirs]
mapping_paths = [glob.glob(f"{ldd}/*mapping.json")[0] for ldd in local_dataset_dirs]


# create global class-area proportion per patch csv...
areas = []
for path in area_paths:
    df = pd.read_csv(path)
    areas.append(df)
df_areas = pd.concat(areas, ignore_index=True)
df_areas.to_csv(os.path.join(DATASET_DIR, GLOBAL_AREAS_BASE), index=False)


# create global patches geospatial vector polygon GeoJSON...
patches = []
for path in patch_paths:
    gdf = gpd.read_file(path)
    patches.append(gdf)
gdf_areas = pd.concat(patches, ignore_index=True)
gdf_areas.to_file(os.path.join(DATASET_DIR, GLOBAL_PATCHES_BASE), driver='GeoJSON', index=False)


# # create global class mapping json...
# class_list = []
# for path in mapping_paths:
#     with open(path, "r") as f:
#         data = json.load(f)
#         class_list.extend(list(data.keys()))
# class_list = list(set(class_list))

# class_dict = {}
# for k, sg in enumerate(class_list, start=1):
#     class_dict[sg] = k
# with open(os.path.join(DATASET_DIR, GLOBAL_MAPPING_BASE), "w") as f:
#     json.dump(class_dict, f, indent=4)


## *Patch Statistics*

In [ ]:

# paths to patch directories
local_dataset_dirs = [os.path.join(DATASET_DIR, d) for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d)) and d != 'smoke']

# iterate through patch_directories...
patch_dirs = []
for d in local_dataset_dirs:
    subdirs = [os.path.join(d, sub) for sub in os.listdir(d) if os.path.isdir(os.path.join(d, sub))]
    patch_dirs.extend(subdirs)


df_stats = pd.DataFrame()


# iterate through modalities...
for mod_name, channels in MODALITIES.items():

    # skip categorical channels...
    if mod_name in ['osm', 'nhd', 'mask']:
        continue
    
    # iterate through moddality channels...
    for c in channels:

        img_paths = []
        for pdir in patch_dirs:
            img_paths.extend(glob.glob(f"{pdir}/*_{c}"))

        pixel_count = 0.0
        pixel_sum = 0.0
        pixel_sum2 = 0.0

        # iterate through image channel paths...
        for ip in img_paths:
            with rasterio.open(ip) as src:
                data = src.read(1, masked=True)
                vals = data.compressed()

                pixel_count += vals.size
                pixel_sum += vals.sum()
                pixel_sum2 += (vals**2).sum()

        mean = pixel_sum / pixel_count
        var = pixel_sum2 / pixel_count - mean**2
        sd = np.sqrt(var)

        df_stats.loc[c, 'mean'] = mean
        df_stats.loc[c, 'sd'] = sd

df_stats.to_csv(os.path.join(DATASET_DIR, GLOBAL_STATS_BASE), index=True)
